<div style="color:	#CC5500;margin:0;font-size:40px;font-family:'Trebuchet MS';text-align:center;display:fill;border-radius:5px;overflow:hidden;font-weight:600;"> California House Price Prediction:</div>


<div style="color:#F88379;margin:0;font-size:40px;font-family:'Trebuchet MS';text-align:center;display:fill;border-radius:5px;overflow:hidden;font-weight:600;"> Tabular Playground Season 3 Episode 1</div>

<div align='center'>
<figure>
  <img src="https://calcog.org/wp-content/uploads/2020/08/1.jpg" alt="Trulli" style="width=100%">
</figure>
</div>

## Environment Setup

In [ ]:
import numpy as np
import pandas as pd 
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
import tensorflow.keras
from keras.models import Sequential
from keras.layers import Dense
import seaborn as sns
import matplotlib.pyplot as plt
from numpy import unique
from numpy import where
from sklearn.datasets import make_classification
from sklearn.cluster import AffinityPropagation
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import Birch
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
import math
from sklearn.ensemble import RandomForestRegressor
sns.set(rc={'figure.figsize':(14,8)})

In [ ]:
def RMSE(y, y_hat):
    return np.sqrt(mean_squared_error(y,y_hat))

In [ ]:
train_df = pd.read_csv('/kaggle/input/playground-series-s3e1/train.csv')
test_df = pd.read_csv('/kaggle/input/playground-series-s3e1/test.csv')
sample_sub = pd.read_csv('/kaggle/input/playground-series-s3e1/sample_submission.csv')

In [ ]:
sample_sub.head()

In [ ]:
train_df.head()

In [ ]:
train_df.shape

In [ ]:
test_df.head()

In [ ]:
test_df.shape

In [ ]:
scaler = StandardScaler()

In [ ]:
mm_scale = MinMaxScaler()

# EDA & Variable Transforms

In [ ]:
corr = train_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
f, ax = plt.subplots(figsize=(11, 9))
cmap = sns.diverging_palette(220, 15, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5})

In [ ]:
sns.scatterplot(data=train_df, x="Latitude", y="Longitude", hue='MedHouseVal')
plt.legend([],[], frameon=False)
plt.title('Latitude, Longitude, & Price')

# Target Variable

In [ ]:
train_df.MedHouseVal.hist()

In [ ]:
#Outlier Removal
def removeOutliers(data, col):
    Q3 = np.quantile(data[col], 0.75)
    Q1 = np.quantile(data[col], 0.25)
    IQR = Q3 - Q1
 
    print("IQR value for column %s is: %s" % (col, IQR))
 
    lower_range = Q1 - 1.5 * IQR
    upper_range = Q3 + 1.5 * IQR
    outlier_free_list = [x for x in data[col] if (
        (x > lower_range) & (x < upper_range))]
    filtered_data = data.loc[data[col].isin(outlier_free_list)]
    return filtered_data

In [ ]:
#Remove target variable outliers
train_df = removeOutliers(train_df, 'MedHouseVal')

In [ ]:
train_df.MedHouseVal.hist()

In [ ]:
train_df.MedHouseVal = train_df.MedHouseVal.apply(lambda x: np.log(x))

In [ ]:
train_df.MedHouseVal.hist()

# Median Income

In [ ]:
train_df.MedInc.hist()

In [ ]:
sns.scatterplot(data=train_df, x="MedInc", y="MedHouseVal")
plt.legend([],[], frameon=False)

In [ ]:
#Drop Outliers over 14
train_df=train_df.drop(train_df[train_df['MedInc']>14].index)

In [ ]:
train_df.MedInc = train_df.MedInc.apply(lambda x: np.log(x))
test_df.MedInc = test_df.MedInc.apply(lambda x: np.log(x))

In [ ]:
train_df.MedInc = scaler.fit_transform(train_df.MedInc.values.reshape(-1, 1))
test_df.MedInc = scaler.fit_transform(test_df.MedInc.values.reshape(-1, 1))

In [ ]:
train_df.MedInc.hist()

In [ ]:
sns.scatterplot(data=train_df, x="MedInc", y="MedHouseVal")
plt.legend([],[], frameon=False)

# House Age

In [ ]:
train_df.HouseAge.hist()

In [ ]:
sns.scatterplot(data=train_df, x="HouseAge", y="MedHouseVal")
plt.legend([],[], frameon=False)

In [ ]:
train_df.HouseAge = scaler.fit_transform(train_df.HouseAge.values.reshape(-1, 1))
test_df.HouseAge = scaler.fit_transform(test_df.HouseAge.values.reshape(-1, 1))

In [ ]:
train_df.HouseAge.hist()

# Average Number of Rooms

In [ ]:
train_df.AveRooms.hist()

In [ ]:
sns.scatterplot(data=train_df, x="AveRooms", y="MedHouseVal")
plt.legend([],[], frameon=False)

In [ ]:
train_df['AvRoom_5_Ind'] = train_df['AveRooms'] > 5

In [ ]:
train_df=train_df.drop(train_df[train_df['AveRooms']>10].index)

In [ ]:
train_df.AveRooms = scaler.fit_transform(train_df.AveRooms.values.reshape(-1, 1))
test_df.AveRooms = scaler.fit_transform(test_df.AveRooms.values.reshape(-1, 1))

In [ ]:
train_df.AveRooms.hist()

# Population

In [ ]:
train_df.Population.hist()

In [ ]:
train_df['Pop_5000_Ind'] = train_df['Population'] > 5000

In [ ]:
sns.scatterplot(data=train_df, x="Population", y="MedHouseVal")
plt.legend([],[], frameon=False)

In [ ]:
train_df=train_df.drop(train_df[train_df['AveRooms']>20000].index)

In [ ]:
train_df.Population = mm_scale.fit_transform(train_df.Population.values.reshape(-1, 1))
test_df.Population = mm_scale.fit_transform(test_df.Population.values.reshape(-1, 1))

In [ ]:
train_df.Population.hist()

# Average Occupancy

In [ ]:
train_df.AveOccup.hist()

In [ ]:
sns.scatterplot(data=train_df, x="AveOccup", y="MedHouseVal")
plt.legend([],[], frameon=False)

In [ ]:
train_df=train_df.drop(train_df[train_df['AveOccup']>50].index)

In [ ]:
train_df.AveOccup = mm_scale.fit_transform(train_df.AveOccup.values.reshape(-1, 1))
test_df.AveOccup = mm_scale.fit_transform(test_df.AveOccup.values.reshape(-1, 1))

In [ ]:
sns.scatterplot(data=train_df, x="AveOccup", y="MedHouseVal")
plt.legend([],[], frameon=False)

# AveBedrms

In [ ]:
train_df.AveBedrms.hist()

In [ ]:
sns.scatterplot(data=train_df, x="AveBedrms", y="MedHouseVal")
plt.legend([],[], frameon=False)

In [ ]:
train_df=train_df.drop(train_df[train_df['AveBedrms']>2.0].index)

In [ ]:
train_df['Bed_12_Ind']=train_df['AveBedrms'] > 1.2

In [ ]:
sns.scatterplot(data=train_df, x="AveBedrms", y="MedHouseVal")
plt.legend([],[], frameon=False)

In [ ]:
train_df.AveBedrms = mm_scale.fit_transform(train_df.AveBedrms.values.reshape(-1, 1))
test_df.AveBedrms = mm_scale.fit_transform(test_df.AveBedrms.values.reshape(-1, 1))

In [ ]:
train_df.head()

In [ ]:
train_df.shape

In [ ]:
train_df['AvRoom_5_Ind'] = train_df['AvRoom_5_Ind'].astype(int)
train_df['Pop_5000_Ind'] = train_df['Pop_5000_Ind'].astype(int)
train_df['Bed_12_Ind'] = train_df['Bed_12_Ind'].astype(int)

# Latitude

In [ ]:
train_df.Latitude.hist()

In [ ]:
sns.scatterplot(data=train_df, x="Latitude", y="MedHouseVal")
plt.legend([],[], frameon=False)

# Longitude

In [ ]:
train_df.Longitude.hist()

In [ ]:
sns.scatterplot(data=train_df, x="Longitude", y="MedHouseVal")
plt.legend([],[], frameon=False)

# 3D Clustering, Geographic Data & House Value

Find centroids, distance to those centroids, and assigned clusters

In [ ]:
train_df.head()

In [ ]:
ax = plt.axes(projection='3d')

# Data for a three-dimensional line

ax.scatter3D(train_df['Latitude'], train_df['Longitude'], train_df['MedHouseVal'], c=train_df['MedHouseVal'], cmap='Greens')

In [ ]:
train_df.Longitude = scaler.fit_transform(train_df.Longitude.values.reshape(-1, 1))
test_df.Longitude = scaler.fit_transform(test_df.Longitude.values.reshape(-1, 1))

In [ ]:
train_df.Latitude = scaler.fit_transform(train_df.Latitude.values.reshape(-1, 1))
test_df.Latitude = scaler.fit_transform(test_df.Latitude.values.reshape(-1, 1))

## KMeans

In [ ]:
KData = train_df[['Latitude', 'Longitude', 'MedHouseVal']]

In [ ]:
wcss = []
for i in range(1,11):
    k_means = KMeans(n_clusters=i,init='k-means++', random_state=42)
    k_means.fit(KData)
    wcss.append(k_means.inertia_)
#plot elbow curve
plt.plot(np.arange(1,11),wcss)
plt.xlabel('Clusters')
plt.ylabel('SSE')
plt.title('')
plt.show()

In [ ]:
k_means_optimum = KMeans(n_clusters = 2, init = 'k-means++',  random_state=42)
y = k_means_optimum.fit_predict(KData)
print(y)

In [ ]:
train_df['cluster'] = y

In [ ]:
#K Means Centroids
c0 = k_means_optimum.cluster_centers_[0][:2]
c1 = k_means_optimum.cluster_centers_[1][:2]

In [ ]:
mask = train_df['cluster'] == 0
train_df.loc[mask, 'c0'] = c0[0] 
train_df.loc[mask, 'c1'] = c0[1] 

In [ ]:
mask = train_df['cluster'] == 1
train_df.loc[mask, 'c0'] = c1[0] 
train_df.loc[mask, 'c1'] = c1[1] 

In [ ]:
import math

In [ ]:
def dist_calc(row):
    x1 = row['Latitude']
    x2 = row['c0']
    y1 = row['Longitude']
    y2 = row['c1']
    dist = math.hypot(x2 - x1, y2 - y1)
    return dist

In [ ]:
train_df['distance'] = train_df.apply(lambda row: dist_calc(row), axis=1)

In [ ]:
def test_dist_calc(row):
    x1 = row['Latitude']
    y1 = row['Longitude']
    dist0 = math.hypot(c0[0] - x1, c0[1] - y1)
    dist1 = math.hypot(c1[0] - x1, c1[1] - y1)
    if dist0 < dist1:
        return dist0
    return dist1

In [ ]:
def test_cluster_calc(row):
    x1 = row['Latitude']
    y1 = row['Longitude']
    dist0 = math.hypot(c0[0] - x1, c0[1] - y1)
    dist1 = math.hypot(c1[0] - x1, c1[1] - y1)
    if dist0 < dist1:
        return 0
    return 1

In [ ]:
test_df['distance'] = test_df.apply(lambda row: test_dist_calc(row), axis=1)
test_df['cluster'] = test_df.apply(lambda row: test_cluster_calc(row), axis=1)

In [ ]:
train_df.distance = scaler.fit_transform(train_df.distance.values.reshape(-1, 1))
test_df.distance = scaler.fit_transform(test_df.distance.values.reshape(-1, 1))

In [ ]:
train_df.cluster = mm_scale.fit_transform(train_df.cluster.values.reshape(-1, 1))
test_df.cluster = mm_scale.fit_transform(test_df.cluster.values.reshape(-1, 1))

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
sns.scatterplot(data=train_df, x="Latitude", y="Longitude", hue='cluster')
plt.legend([],[], frameon=False)
plt.title("K Means Predicted Clusters (Training)")

In [ ]:
sns.scatterplot(data=test_df, x="Latitude", y="Longitude", hue='cluster')
plt.legend([],[], frameon=False)
plt.title("K Means Predicted Clusters (Test)")

In [ ]:
KData = train_df[['Longitude', 'MedHouseVal']]

In [ ]:
#Evaluate number of clusters without Latitude
wcss = []
for i in range(1,11):
    k_means = KMeans(n_clusters=i,init='k-means++', random_state=42)
    k_means.fit(KData)
    wcss.append(k_means.inertia_)
#plot elbow curve
plt.plot(np.arange(1,11),wcss)
plt.xlabel('Clusters')
plt.ylabel('SSE')
plt.show()

In [ ]:
KData = train_df[['Latitude', 'MedHouseVal']]

In [ ]:
#Evaluate number of clusters without Longitude
wcss = []
for i in range(1,11):
    k_means = KMeans(n_clusters=i,init='k-means++', random_state=42)
    k_means.fit(KData)
    wcss.append(k_means.inertia_)
#plot elbow curve
plt.plot(np.arange(1,11),wcss)
plt.xlabel('Clusters')
plt.ylabel('SSE')
plt.show()

## BIRCH Clustering

In [ ]:
CData = train_df[['Latitude', 'Longitude', 'MedHouseVal']]
birch_model = Birch(threshold=0.01, n_clusters=6)
# fit the model
birch_model.fit(CData)
# assign a cluster to each example
yhat = birch_model.predict(CData)
CData['cluster'] = yhat
# retrieve unique clusters
clusters = unique(yhat)
# create scatter plot for samples from each cluster
sns.scatterplot(data=CData, x="Latitude", y="Longitude", hue='cluster')
plt.legend([],[], frameon=False)
# show the plot
plt.show()

In [ ]:
def birch_predict(model, X):

    nr_samples = X.shape[0]

    y_new = np.ones(shape=nr_samples, dtype=int) * -1
    
    num_centers = len(birch_model.subcluster_centers_)
    
    centers = birch_model.subcluster_centers_[:,:2]

    for i in range(nr_samples):
        
        point_rep = np.column_stack((np.ones(shape=num_centers, dtype=int) * X.loc[i, 'Latitude'], np.ones(shape=num_centers, dtype=int) * X.loc[i, 'Longitude']))
        
        diff = birch_model.subcluster_centers_[:,:2] - point_rep # NumPy broadcasting

        dist = np.linalg.norm(diff, axis=1)  # Euclidean distance

        shortest_dist_idx = np.argmin(dist)

        y_new[i] = model.subcluster_labels_[shortest_dist_idx]

    return y_new

In [ ]:
train_df['price_cat'] = yhat

In [ ]:
birch_model.subcluster_centers_[:,:2]

In [ ]:
num_centers = len(birch_model.subcluster_labels_)

In [ ]:
len(birch_model.labels_)

In [ ]:
test_df['price_cat'] = birch_predict(birch_model, test_df[['Latitude', 'Longitude']])

In [ ]:
test_df.price_cat.unique()

In [ ]:
train_df.price_cat = scaler.fit_transform(train_df.price_cat.values.reshape(-1, 1))
test_df.price_cat = scaler.fit_transform(test_df.price_cat.values.reshape(-1, 1))

## DBSCAN

In [ ]:
# DBSCAN


db_model = DBSCAN(eps=0.30, min_samples=9)
# fit model and predict clusters
yhat = db_model.fit_predict(CData)
# retrieve unique clusters
clusters = unique(yhat)
# create scatter plot for samples from each cluster
CData['cluster'] = yhat
# retrieve unique clusters
clusters = unique(yhat)
# create scatter plot for samples from each cluster
sns.scatterplot(data=CData, x="Latitude", y="Longitude", hue='cluster')
plt.legend([],[], frameon=False)
plt.show()

In [ ]:
clusters

In [ ]:
np.unique(db_model.labels_)

In [ ]:
len(db_model.core_sample_indices_)

In [ ]:
#Predict clusters based only on latitude and longitude dimensions
def dbscan_predict(model, X):

    nr_samples = X.shape[0]

    y_new = np.ones(shape=nr_samples, dtype=int) * -1
    
    num_centers = len(model.components_)

    for i in range(nr_samples):
        
        point_rep = np.column_stack((np.ones(shape=num_centers, dtype=int) * X.loc[i, 'Latitude'], np.ones(shape=num_centers, dtype=int) * X.loc[i, 'Longitude']))
        
        diff = model.components_[:,:2] - point_rep

        dist = np.linalg.norm(diff, axis=1)  # Euclidean distance

        shortest_dist_idx = np.argmin(dist)

        y_new[i] = model.labels_[shortest_dist_idx]

    return y_new

In [ ]:
train_df['scan_cluster'] = yhat

In [ ]:
train_df.scan_cluster.unique()

In [ ]:
test_df['scan_cluster'] = dbscan_predict(db_model, test_df[['Latitude', 'Longitude']])

In [ ]:
test_df.scan_cluster.unique()

In [ ]:
train_df.scan_cluster = scaler.fit_transform(train_df.scan_cluster.values.reshape(-1, 1))
test_df.scan_cluster = scaler.fit_transform(test_df.scan_cluster.values.reshape(-1, 1))

In [ ]:
test_df.head()

# Data Split

In [ ]:
train_cols = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'Bed_12_Ind', 'Pop_5000_Ind', 'AvRoom_5_Ind']

y_col = ['MedHouseVal']

In [ ]:
#Most predictive variables according to random forest feature importance
base_cols = ['MedInc', 'HouseAge', 'AveRooms', 'AveOccup', 'Latitude', 'Longitude']

In [ ]:
train_cols = ['MedInc', 'HouseAge', 'AveRooms', 'AveOccup', 'Latitude', 'Longitude', 'cluster', 'distance', 'price_cat', 'scan_cluster']

In [ ]:
X = train_df[train_cols]
y = train_df[y_col]

In [ ]:
#test train split
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                   random_state=104, 
                                   test_size=0.25, 
                                   shuffle=True)

In [ ]:
X_base = train_df[base_cols]

Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_base,y,
                                   random_state=104, 
                                   test_size=0.25, 
                                   shuffle=True)

In [ ]:
rfr = RandomForestRegressor(max_depth=5, random_state=0, n_estimators=500)
rfr.fit(Xb_train, yb_train)

yb_pred = rfr.predict(Xb_test)
print(RMSE(yb_test,yb_pred))
print(mean_absolute_error(yb_test, yb_pred))

In [ ]:
importances = rfr.feature_importances_
feature_names = [f"{i}" for i in X_base.columns]

forest_importances = pd.Series(importances, index=feature_names)
std = np.std([tree.feature_importances_ for tree in rfr.estimators_], axis=0)


fig, ax = plt.subplots()
forest_importances.plot.bar(yerr=std, ax=ax)
ax.set_title(f"Y = MedHouseVal, Feature Importance")
ax.set_ylabel("Mean decrease in impurity")
ax.set_xlabel("Feature")
fig.tight_layout()

# XGBoost

In [ ]:
import xgboost as xgb

In [ ]:
params_xgb = {'n_estimators': 1000, 'random_state':0}

In [ ]:
xgb_mod = xgb.XGBRegressor(
        **params_xgb,
        objective='reg:squarederror')
#model.fit(train_s, y_train)

In [ ]:
xgb_mod.fit(X_train, y_train)

In [ ]:
y_pred = xgb_mod.predict(X_test)
print(RMSE(y_test,y_pred))
print(mean_absolute_error(y_test, y_pred))

In [ ]:
train_df.shape

# Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
linear = LinearRegression()
linear.fit(X_train, y_train)

In [ ]:
y_pred = linear.predict(X_test)
print(RMSE(y_test,y_pred))
print(mean_absolute_error(y_test, y_pred))

# LGBM

In [ ]:
from lightgbm import LGBMRegressor

In [ ]:
params_lgb = {
    "n_estimators": 1000,
    "verbose": -1
}

In [ ]:
model_lgbm = LGBMRegressor(**params_lgb)
model_lgbm.fit(X_train, y_train)

In [ ]:
y_pred = model_lgbm.predict(X_test)
print(RMSE(y_test,y_pred))
print(mean_absolute_error(y_test, y_pred))

# Neural Network

In [ ]:
X_train.head()

In [ ]:
model = Sequential()
 
model.add(Dense(units=512, input_dim=10, kernel_initializer='normal', activation='relu'))

model.add(Dense(units=100, kernel_initializer='normal', activation='tanh'))

model.add(Dense(1, kernel_initializer='normal'))

model.compile(loss='mean_squared_error', optimizer='adam')

model.fit(X_train, y_train ,batch_size = 20, epochs = 50, verbose=1)

In [ ]:
y_pred = model.predict(X_test)
print(RMSE(np.exp(y_test),np.exp(y_pred)))
print(mean_absolute_error(y_test, y_pred))

# CatBoost

In [ ]:
from catboost import CatBoostRegressor

In [ ]:
cat = CatBoostRegressor(iterations=5000,
                        verbose = 500,
                        eval_metric='RMSE',
                        max_depth = 6,
                        subsample=0.7,
                        learning_rate = 0.04)
cat.fit(X_train, y_train, eval_set=[(X_test, y_test)], early_stopping_rounds=1000)

In [ ]:
y_pred = cat.predict(X_test)
print(RMSE(y_test,y_pred))
print(mean_absolute_error(y_test, y_pred))

In [ ]:
y_pred = cat.predict(X_test)
print(RMSE(np.exp(y_test),np.exp(y_pred)))
print(mean_absolute_error(y_test, y_pred))

# Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
ridge = Ridge(copy_X=False)
ridge.fit(X_train, y_train)

In [ ]:
y_pred = ridge.predict(X_test)
print(RMSE(y_test,y_pred))
print(mean_absolute_error(y_test, y_pred))

# SGD

In [ ]:
from sklearn.linear_model import SGDRegressor

In [ ]:
sgd = SGDRegressor(max_iter=1000, tol=1e-3)
sgd.fit(X_train, y_train)

In [ ]:
y_pred = sgd.predict(X_test)
print(RMSE(y_test,y_pred))
print(mean_absolute_error(y_test, y_pred))

In [ ]:
from sklearn.linear_model import BayesianRidge

In [ ]:
bay_ridge = BayesianRidge()
bay_ridge.fit(X_train, y_train)

In [ ]:
y_pred = bay_ridge.predict(X_test)
print(RMSE(y_test,y_pred))
print(mean_absolute_error(y_test, y_pred))

# Random Forest Regressor

In [ ]:
rfr = RandomForestRegressor(max_depth=5, random_state=0, n_estimators=500)
rfr.fit(X_train, y_train)

In [ ]:
y_pred = rfr.predict(X_test)
print(RMSE(y_test,y_pred))
print(mean_absolute_error(y_test, y_pred))

In [ ]:
importances = rfr.feature_importances_
feature_names = [f"{i}" for i in X.columns]

forest_importances = pd.Series(importances, index=feature_names)
std = np.std([tree.feature_importances_ for tree in rfr.estimators_], axis=0)


fig, ax = plt.subplots()
forest_importances.plot.bar(yerr=std, ax=ax)
ax.set_title(f"Y = MedHouseVal, Feature Importance")
ax.set_ylabel("Mean decrease in impurity")
ax.set_xlabel("Feature")
fig.tight_layout()

# SVM

In [ ]:
from sklearn.svm import SVR

In [ ]:
svr = SVR(C=1.0, epsilon=0.2)
svr.fit(X_train, y_train)

In [ ]:
y_pred = svr.predict(X_test)
print(RMSE(y_test,y_pred))
print(mean_absolute_error(y_test, y_pred))

# Stacked Model

In [ ]:
first_pred_1 = model.predict(X_train)
first_pred_2 = xgb_mod.predict(X_train)
first_pred_3 = model_lgbm.predict(X_train)
first_pred_4 = svr.predict(X_train)
first_pred_5 = cat.predict(X_train)

stack_pred = np.column_stack((first_pred_1,first_pred_2,first_pred_3,first_pred_4, first_pred_5))

In [ ]:
params = {'learning_rate': 0.3, 
          'depth': 12, 
          'l2_leaf_reg': 4, 
          'loss_function': 'RMSE', 
          'eval_metric': 'RMSE', 
          'task_type': 'CPU', 
          'iterations': 40,
          'od_type': 'Iter', 
          'boosting_type': 'Plain', 
          'bootstrap_type': 'Bayesian', 
          'allow_const_label': True, 
          'random_state': 1
         }

In [ ]:
meta_model =  CatBoostRegressor(**params)

meta_model.fit(stack_pred, y_train)

# Submission Construction

In [ ]:
#Meta Model
first_pred_1 = model.predict(test_df[train_cols])
first_pred_2 = xgb_mod.predict(test_df[train_cols])
first_pred_3 = model_lgbm.predict(test_df[train_cols])
first_pred_4 = svr.predict(test_df[train_cols])
first_pred_5 = cat.predict(test_df[train_cols])




stack_pred = np.column_stack((first_pred_1,first_pred_2,first_pred_3,first_pred_4,first_pred_5))

In [ ]:
#test_df['MedHouseVal'] = np.exp(meta_model.predict(stack_pred))
test_df['MedHouseVal'] = np.exp(cat.predict(test_df[train_cols]))

In [ ]:
test_df['lgbm_pred'] = model_lgbm.predict(test_df[train_cols])
test_df['cat_pred'] = cat.predict(test_df[train_cols])

In [ ]:
#just CATBOOST model
#test_df['MedHouseVal'] = test_df.apply(lambda x: np.exp(x['cat_pred']), axis=1)

In [ ]:
test_df['MedHouseVal'] = test_df.apply(lambda x: np.exp((x['lgbm_pred'] + x['cat_pred'])/2), axis=1)

In [ ]:
test_df.head()

In [ ]:
submission = test_df[['id', 'MedHouseVal']]

In [ ]:
submission.to_csv('submission.csv', index=False)

# Conclusion



The model that recieved the best score (0.654) was my simple average ensemble of the LGBM and CatBoost regressors without any added features from clustering. The clustering columns helped these two models perform better on the validation set but as can be seen by the increased out of sample RMSE from the neural network (despite the better loss measure) this led to a massive overfitting problem. 